In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path().resolve().parents[0]

METADATA_PATH = PROJECT_ROOT / "data" / "interim" / "trap_metadata.csv"
TRAP_COUNT_PATH = PROJECT_ROOT / "data" / "processed" / "stopwhitefly_trap_weekly_counts_with_metadata.csv"

metadata_df = pd.read_csv(METADATA_PATH)
trap_count_df = pd.read_csv(TRAP_COUNT_PATH)

print("Metadata shape:", metadata_df.shape)
print("Trap-count shape:", trap_count_df.shape)

print("\nMetadata columns:")
print(metadata_df.columns.tolist())

print("\nTrap-count columns:")
print(trap_count_df.columns.tolist())

print("\nMetadata dtypes:")
print(metadata_df.dtypes)

print("\nTrap-week dtypes:")
print(trap_count_df.dtypes)

print("\nMetadata missing values:")
print(metadata_df.isna().sum())

print("\nTrap-count missing values:")
print(trap_count_df.isna().sum())

print("\nUnique metadata sites:", metadata_df["site_id"].nunique())
print("Unique trap-week sites:", trap_count_df["site_id"].nunique())

print("\nTrap years:")
print(sorted(trap_count_df["year"].unique()))

print("\nLatitude range:", metadata_df["latitude"].min(), metadata_df["latitude"].max())
print("Longitude range:", metadata_df["longitude"].min(), metadata_df["longitude"].max())

metadata_df.head()

Metadata shape: (31, 7)
Trap-count shape: (6121, 11)

Metadata columns:
['site_id', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']

Trap-count columns:
['site_id', 'year', 'plotted_date', 'date_collected', 'whitefly_count', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']

Metadata dtypes:
site_id           int64
trap_label          str
latitude        float64
longitude       float64
status              str
first_report        str
last_report         str
dtype: object

Trap-week dtypes:
site_id             int64
year                int64
plotted_date          str
date_collected        str
whitefly_count      int64
trap_label            str
latitude          float64
longitude         float64
status                str
first_report          str
last_report           str
dtype: object

Metadata missing values:
site_id         0
trap_label      0
latitude        0
longitude       0
status          0
first_report    0
last_repor

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


#### Convert the trap metadata from pandas dataframe to GeoDataframe

In [3]:
import geopandas as gpd

In [5]:
trap_gdf = gpd.GeoDataFrame(
    metadata_df.copy(),
    geometry=gpd.points_from_xy(metadata_df["longitude"], metadata_df["latitude"]),
    crs="EPSG:4326"  # geographic CRS (points are in geographic degrees)
)

print("Original CRS:", trap_gdf.crs)
print("Shape:", trap_gdf.shape)

print("\nGeometry type counts:")
print(trap_gdf.geometry.geom_type.value_counts())

print("\nBounds in latitude/longitude CRS:")
print(trap_gdf.total_bounds)

trap_gdf.head()

Original CRS: EPSG:4326
Shape: (31, 8)

Geometry type counts:
Point    31
Name: count, dtype: int64

Bounds in latitude/longitude CRS:
[-84.895381  30.798771 -83.405028  31.557827]


,site_id,trap_label,latitude,longitude,status,first_report,last_report,geometry
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22,POINT (-83.46501 31.46603)
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22,POINT (-83.40503 31.46509)
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22,POINT (-83.48018 31.55783)
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15,POINT (-83.57304 31.54121)
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15,POINT (-83.66288 31.52428)


#### Reproject into a meter-based CRS

In [7]:
# Reproject trap points from geographic CRS to projected CRS (UTM Zone 17N)
PROJECTED_CRS = "EPSG:26917"

trap_gdf_utm = trap_gdf.to_crs(PROJECTED_CRS)

print("Original CRS:", trap_gdf.crs)
print("Projected CRS:", trap_gdf_utm.crs)

print("\nBounds in projected CRS/meters:")
print(trap_gdf_utm.total_bounds)

print("\nFirst few projected geometries:")
print(trap_gdf_utm[["site_id", "trap_label", "geometry"]].head())

Original CRS: EPSG:4326
Projected CRS: EPSG:26917

Bounds in projected CRS/meters:
[ 129436.91781993 3410393.01188487  271492.41530177 3494093.91594691]

First few projected geometries:
   site_id trap_label                        geometry
0    29221   Trap 101  POINT (265794.327 3483882.818)
1    29222   Trap 102  POINT (271492.415 3483651.729)
2    29224   Trap 104  POINT (264582.565 3494093.916)
3    29225   Trap 105  POINT (255722.656 3492455.271)
4    29226   Trap 106  POINT (247145.224 3490781.095)


#### Create 500 m and 1 km buffers

In [10]:
# create buffer layers in projected CRS
trap_buffers_500m = trap_gdf_utm.copy()
trap_buffers_500m["buffer_radius_m"] = 500
trap_buffers_500m["geometry"] = trap_buffers_500m.geometry.buffer(500)

trap_buffers_1km = trap_gdf_utm.copy()
trap_buffers_1km["buffer_radius_m"] = 1000
trap_buffers_1km["geometry"] = trap_buffers_1km.geometry.buffer(1000)

print("500 m buffer CRS:", trap_buffers_500m.crs)
print("1 km buffer CRS:", trap_buffers_1km.crs)

print("\n500 m buffer shape:", trap_buffers_500m.shape)
print("1 km buffer shape:", trap_buffers_1km.shape)

print("\nGeometry types:")
print("500 m:")
print(trap_buffers_500m.geometry.geom_type.value_counts())
print("\n1 km:")
print(trap_buffers_1km.geometry.geom_type.value_counts())

print("\nApproximate buffer areas in square meters:")
print("500 m buffer area, first site:", trap_buffers_500m.geometry.area.iloc[0])
print("1 km buffer area, first site:", trap_buffers_1km.geometry.area.iloc[0])

500 m buffer CRS: EPSG:26917
1 km buffer CRS: EPSG:26917

500 m buffer shape: (31, 9)
1 km buffer shape: (31, 9)

Geometry types:
500 m:
Polygon    31
Name: count, dtype: int64

1 km:
Polygon    31
Name: count, dtype: int64

Approximate buffer areas in square meters:
500 m buffer area, first site: 784137.1226364922
1 km buffer area, first site: 3136548.490546164


#### Combining the two buffer layers

In [11]:
trap_buffers_all = pd.concat(
    [trap_buffers_500m, trap_buffers_1km],
    ignore_index=True
)

print("Combined buffer shape:", trap_buffers_all.shape)
print("\nBuffer radius counts:")
print(trap_buffers_all["buffer_radius_m"].value_counts().sort_index())

print("\nCRS:", trap_buffers_all.crs)

print("\nGeometry type counts:")
print(trap_buffers_all.geometry.geom_type.value_counts())

trap_buffers_all[["site_id", "trap_label", "buffer_radius_m", "geometry"]].head()

Combined buffer shape: (62, 9)

Buffer radius counts:
buffer_radius_m
500     31
1000    31
Name: count, dtype: int64

CRS: EPSG:26917

Geometry type counts:
Polygon    62
Name: count, dtype: int64


,site_id,trap_label,buffer_radius_m,geometry
0,29221,Trap 101,500,"POLYGON ((266294.327 3483882.818, 266291.919 3..."
1,29222,Trap 102,500,"POLYGON ((271992.415 3483651.729, 271990.008 3..."
2,29224,Trap 104,500,"POLYGON ((265082.565 3494093.916, 265080.158 3..."
3,29225,Trap 105,500,"POLYGON ((256222.656 3492455.271, 256220.248 3..."
4,29226,Trap 106,500,"POLYGON ((247645.224 3490781.095, 247642.816 3..."


#### Save buffer layer for reuse

In [12]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "geospatial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUFFER_OUTPUT_PATH = OUTPUT_DIR / "trap_buffers_500m_1km.gpkg"

# Save as a GeoPackage (.gpkg)
trap_buffers_all.to_file(BUFFER_OUTPUT_PATH, layer="trap_buffers", driver="GPKG")

print("Saved buffer layer to:", BUFFER_OUTPUT_PATH)

Saved buffer layer to: /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/interim/geospatial/trap_buffers_500m_1km.gpkg


#### Validation to ensure saved GeoPackage can be read correctly

In [13]:
buffers_check = gpd.read_file(BUFFER_OUTPUT_PATH, layer="trap_buffers")

print("Shape of saved file:", buffers_check.shape)
print("CRS of saved file:", buffers_check.crs)

print("\nBuffer radius counts:")
print(buffers_check["buffer_radius_m"].value_counts().sort_index())

print("\nGeometry type counts:")
print(buffers_check.geometry.geom_type.value_counts())

print("\nColumns:")
print(buffers_check.columns.to_list())

buffers_check[["site_id", "trap_label", "buffer_radius_m", "geometry"]].head()

Shape of saved file: (62, 9)
CRS of saved file: EPSG:26917

Buffer radius counts:
buffer_radius_m
500     31
1000    31
Name: count, dtype: int64

Geometry type counts:
Polygon    62
Name: count, dtype: int64

Columns:
['site_id', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report', 'buffer_radius_m', 'geometry']


,site_id,trap_label,buffer_radius_m,geometry
0,29221,Trap 101,500,"POLYGON ((266294.327 3483882.818, 266291.919 3..."
1,29222,Trap 102,500,"POLYGON ((271992.415 3483651.729, 271990.008 3..."
2,29224,Trap 104,500,"POLYGON ((265082.565 3494093.916, 265080.158 3..."
3,29225,Trap 105,500,"POLYGON ((256222.656 3492455.271, 256220.248 3..."
4,29226,Trap 106,500,"POLYGON ((247645.224 3490781.095, 247642.816 3..."


#### Next, we download the land cover raster for the year 2025 for southern Georgia from MRLC website and save the files in the ~/data/raw/nlcd/2025 folder. 

#### Inspect the raster

In [14]:
from pathlib import Path
import rasterio

In [15]:
NLCD_2025_PATH = PROJECT_ROOT / "data" / "raw" / "nlcd" / "2025" / "nlcd_landcover_2025.tif"

with rasterio.open(NLCD_2025_PATH) as src:
    print("Raster path:", NLCD_2025_PATH)
    print("CRS:", src.crs)
    print("Width, height:", src.width, src.height)
    print("Number of bands:", src.count)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Nodata:", src.nodata)
    print("Data type:", src.dtypes)

Raster path: /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2025/nlcd_landcover_2025.tif
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Width, height: 14847 11731
Number of bands: 1
Resolution: (30.0, 30.0)
Bounds: BoundingBox(left=958635.0, bottom=830025.0, right=1404045.0, top=1181955.0)
Nodata: 250.0
Data type: ('uint8',)


#### Verify overlap between buffers and NLCD raster

In [16]:
import geopandas as gpd
import rasterio
from shapely.geometry import box

In [17]:
BUFFER_PATH = PROJECT_ROOT / "data" / "interim" / "geospatial" / "trap_buffers_500m_1km.gpkg"

# Read buffer layer
buffers_gdf = gpd.read_file(BUFFER_PATH, layer="trap_buffers")

# Open the NLCD raster
with rasterio.open(NLCD_2025_PATH) as src:
    nlcd_crs = src.crs
    nlcd_bounds = src.bounds

# Reproject buffers to NLCD CRS
buffers_nlcd_crs = buffers_gdf.to_crs(nlcd_crs)

# Create raster footprint polygon
nlcd_bounds_gdf = gpd.GeoDataFrame(
    {"name": ["nlcd_bounds"]},
    geometry=[box(nlcd_bounds.left, nlcd_bounds.bottom, nlcd_bounds.right, nlcd_bounds.top)],
    crs=nlcd_crs
)

print("Original buffer CRS:", buffers_gdf.crs)
print("NLCD CRS:", nlcd_crs)
print("Reprojected buffer CRS:", buffers_nlcd_crs.crs)

print("\nBuffer bounds in NLCD CRS:")
print(buffers_nlcd_crs.total_bounds)

print("\nNLCD raster bounds:")
print(nlcd_bounds)

print("\nDo all buffers intersect NLCD raster bounds?")
print(buffers_nlcd_crs.intersects(nlcd_bounds_gdf.geometry.iloc[0]).all())

print("\nNumber of buffers intersecting NLCD raster bounds:")
print(buffers_nlcd_crs.intersects(nlcd_bounds_gdf.geometry.iloc[0]).sum(), "of", len(buffers_nlcd_crs))

Original buffer CRS: EPSG:26917
NLCD CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Reprojected buffer CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["lon

#### First extraction function: one raster year, all buffers

In [18]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

In [79]:
records = []
with rasterio.open(NLCD_2025_PATH) as src:
    raster_crs = src.crs
    nodata = src.nodata

    buffers_for_raster = buffers_gdf.to_crs(raster_crs)

    for _, row in buffers_for_raster.iterrows():
        geom = [row.geometry]   # mask() requires a list of geometries

        # Clip the raster to the polygon
        out_image, _ = mask(
            src,                # the raster
            geom,               # the polygon to clip with
            crop=True,          # output only the area inside the polygon
            filled=True,        # fill areas outside the polygon with nodata
            nodata=nodata       # use the raster's nodata value
        )

        # flatten the pixel values in the first and only band in the out_image
        values = out_image[0].ravel()

        # Remove nodata values
        if nodata is not None:
            values = values[values != nodata]

        # Remove zeros when present as unclassified or background
        values = values[values != 0]

        unique_values, counts = np.unique(values, return_counts=True)

        for class_value, counts in zip(unique_values, counts):
            records.append(
                {
                    "site_id": row["site_id"],
                    "trap_label": row["trap_label"],
                    "buffer_radius_m": row["buffer_radius_m"],
                    "nlcd_class": int(class_value),
                    "pixel_count": int(counts)
                }
            )
    nlcd_counts_2025 = pd.DataFrame(records)

display(nlcd_counts_2025.head())

,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count
0,29221,Trap 101,500,11,10
1,29221,Trap 101,500,21,107
2,29221,Trap 101,500,22,70
3,29221,Trap 101,500,42,149
4,29221,Trap 101,500,52,15


In [81]:
print("NLCD class count table shape:", nlcd_counts_2025.shape)
print("\nFirst rows:")
display(nlcd_counts_2025.head(10))

print("\nUnique NLCD classes found:")
print(sorted(nlcd_counts_2025["nlcd_class"].unique()))

print("\nNumber of site-buffer combinations:")
print(nlcd_counts_2025[["site_id", "buffer_radius_m"]].drop_duplicates().shape[0])

NLCD class count table shape: (639, 5)

First rows:


,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count
0,29221,Trap 101,500,11,10
1,29221,Trap 101,500,21,107
2,29221,Trap 101,500,22,70
3,29221,Trap 101,500,42,149
4,29221,Trap 101,500,52,15
5,29221,Trap 101,500,71,8
6,29221,Trap 101,500,82,402
7,29221,Trap 101,500,90,108
8,29221,Trap 101,500,95,2
9,29222,Trap 102,500,11,26



Unique NLCD classes found:
[np.int64(11), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(31), np.int64(41), np.int64(42), np.int64(43), np.int64(52), np.int64(71), np.int64(81), np.int64(82), np.int64(90), np.int64(95)]

Number of site-buffer combinations:
62


#### Convert the long class-count table into grouped biological features

In [93]:
NLCD_CLASS_GROUPS = {
    "water": [11],
    "developed": [21, 22, 23, 24],
    "barren": [31],
    "forest": [41, 42, 43],
    "shrub_scrub": [52],
    "grassland_herbaceous": [71],
    "pasture_hay": [81],
    "cultivated_crops": [82],
    "wetlands": [90, 95],
}

class_to_group = {}
for group, codes in NLCD_CLASS_GROUPS.items():
    for code in codes:
        class_to_group[code] = group

class_to_group

{11: 'water',
 21: 'developed',
 22: 'developed',
 23: 'developed',
 24: 'developed',
 31: 'barren',
 41: 'forest',
 42: 'forest',
 43: 'forest',
 52: 'shrub_scrub',
 71: 'grassland_herbaceous',
 81: 'pasture_hay',
 82: 'cultivated_crops',
 90: 'wetlands',
 95: 'wetlands'}

In [95]:
nlcd_counts_2025["nlcd_class_label"] = nlcd_counts_2025["nlcd_class"].map(class_to_group)
nlcd_counts_2025["nlcd_class_label"] = nlcd_counts_2025["nlcd_class_label"].fillna("other")

nlcd_counts_2025.head()

,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count,nlcd_class_label
0,29221,Trap 101,500,11,10,water
1,29221,Trap 101,500,21,107,developed
2,29221,Trap 101,500,22,70,developed
3,29221,Trap 101,500,42,149,forest
4,29221,Trap 101,500,52,15,shrub_scrub


In [109]:
nlcd_counts_labeled_2025 = nlcd_counts_2025.copy()

print("Labeled counts shape:", nlcd_counts_labeled_2025.shape)

print("\nGroup counts:")
print(nlcd_counts_labeled_2025["nlcd_class_label"].value_counts())

display(nlcd_counts_labeled_2025.head(20))

Labeled counts shape: (639, 6)

Group counts:
nlcd_class_label
developed               160
wetlands                109
forest                  102
cultivated_crops         62
shrub_scrub              55
water                    53
grassland_herbaceous     50
pasture_hay              43
barren                    5
Name: count, dtype: int64


,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count,nlcd_class_label
0,29221,Trap 101,500,11,10,water
1,29221,Trap 101,500,21,107,developed
2,29221,Trap 101,500,22,70,developed
3,29221,Trap 101,500,42,149,forest
4,29221,Trap 101,500,52,15,shrub_scrub
5,29221,Trap 101,500,71,8,grassland_herbaceous
6,29221,Trap 101,500,82,402,cultivated_crops
7,29221,Trap 101,500,90,108,wetlands
8,29221,Trap 101,500,95,2,wetlands
9,29222,Trap 102,500,11,26,water


#### Convert land cover pixel counts to fractions

In [157]:
# Calculate total pixel counts for each site ID x buffer
total_counts = (
    nlcd_counts_labeled_2025.groupby(["site_id", "trap_label", "buffer_radius_m"], as_index=False)
    ["pixel_count"]
    .sum()
    .rename(columns={"pixel_count": "total_valid_pixels"})
)
total_counts.head()

,site_id,trap_label,buffer_radius_m,total_valid_pixels
0,29221,Trap 101,500,871
1,29221,Trap 101,1000,3487
2,29222,Trap 102,500,871
3,29222,Trap 102,1000,3486
4,29224,Trap 104,500,870


In [158]:
# Calculate pixel counts for each grouped land cover class
grouped_counts = (
    nlcd_counts_labeled_2025.groupby(
        ["site_id", "trap_label", "buffer_radius_m", "nlcd_class_label"],
        as_index=False
    )
    ["pixel_count"]
    .sum()
)
grouped_counts.head(10)

,site_id,trap_label,buffer_radius_m,nlcd_class_label,pixel_count
0,29221,Trap 101,500,cultivated_crops,402
1,29221,Trap 101,500,developed,177
2,29221,Trap 101,500,forest,149
3,29221,Trap 101,500,grassland_herbaceous,8
4,29221,Trap 101,500,shrub_scrub,15
5,29221,Trap 101,500,water,10
6,29221,Trap 101,500,wetlands,110
7,29221,Trap 101,1000,cultivated_crops,1742
8,29221,Trap 101,1000,developed,690
9,29221,Trap 101,1000,forest,448


In [160]:
# Merge total counts to the grouped counts
nlcd_counts_merged_2025 = grouped_counts.merge(
    total_counts,
    on=["site_id", "trap_label", "buffer_radius_m"],
    how="left"
)
nlcd_counts_merged_2025.head(20)

,site_id,trap_label,buffer_radius_m,nlcd_class_label,pixel_count,total_valid_pixels
0,29221,Trap 101,500,cultivated_crops,402,871
1,29221,Trap 101,500,developed,177,871
2,29221,Trap 101,500,forest,149,871
3,29221,Trap 101,500,grassland_herbaceous,8,871
4,29221,Trap 101,500,shrub_scrub,15,871
5,29221,Trap 101,500,water,10,871
6,29221,Trap 101,500,wetlands,110,871
7,29221,Trap 101,1000,cultivated_crops,1742,3487
8,29221,Trap 101,1000,developed,690,3487
9,29221,Trap 101,1000,forest,448,3487


In [161]:
nlcd_group_fractions_2025 = nlcd_counts_merged_2025.copy()

nlcd_group_fractions_2025["land_cover_fraction"] = (
    nlcd_group_fractions_2025["pixel_count"] / nlcd_group_fractions_2025["total_valid_pixels"]
)

nlcd_group_fractions_2025.head(10)

,site_id,trap_label,buffer_radius_m,nlcd_class_label,pixel_count,total_valid_pixels,land_cover_fraction
0,29221,Trap 101,500,cultivated_crops,402,871,0.461538
1,29221,Trap 101,500,developed,177,871,0.203215
2,29221,Trap 101,500,forest,149,871,0.171068
3,29221,Trap 101,500,grassland_herbaceous,8,871,0.009185
4,29221,Trap 101,500,shrub_scrub,15,871,0.017222
5,29221,Trap 101,500,water,10,871,0.011481
6,29221,Trap 101,500,wetlands,110,871,0.126292
7,29221,Trap 101,1000,cultivated_crops,1742,3487,0.499570
8,29221,Trap 101,1000,developed,690,3487,0.197878
9,29221,Trap 101,1000,forest,448,3487,0.128477


In [162]:
print("Group fraction table shape:", nlcd_group_fractions_2025.shape)

print("\nFirst rows:")
display(nlcd_group_fractions_2025.head(20))

print("\nFraction sum check:")
fraction_check = (
    nlcd_group_fractions_2025
    .groupby(["site_id", "buffer_radius_m"])["land_cover_fraction"]
    .sum()
    .reset_index(name="fraction_sum")
)

display(fraction_check.head())
print(fraction_check["fraction_sum"].describe())

Group fraction table shape: (454, 7)

First rows:


,site_id,trap_label,buffer_radius_m,nlcd_class_label,pixel_count,total_valid_pixels,land_cover_fraction
0,29221,Trap 101,500,cultivated_crops,402,871,0.461538
1,29221,Trap 101,500,developed,177,871,0.203215
2,29221,Trap 101,500,forest,149,871,0.171068
3,29221,Trap 101,500,grassland_herbaceous,8,871,0.009185
4,29221,Trap 101,500,shrub_scrub,15,871,0.017222
5,29221,Trap 101,500,water,10,871,0.011481
6,29221,Trap 101,500,wetlands,110,871,0.126292
7,29221,Trap 101,1000,cultivated_crops,1742,3487,0.499570
8,29221,Trap 101,1000,developed,690,3487,0.197878
9,29221,Trap 101,1000,forest,448,3487,0.128477



Fraction sum check:


,site_id,buffer_radius_m,fraction_sum
0,29221,500,1.0
1,29221,1000,1.0
2,29222,500,1.0
3,29222,1000,1.0
4,29224,500,1.0


count    6.200000e+01
mean     1.000000e+00
std      1.421495e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: fraction_sum, dtype: float64


#### Create model ready wide table

In [163]:
wide_df = nlcd_group_fractions_2025.pivot_table(
    index=["site_id", "trap_label"],
    columns=["buffer_radius_m", "nlcd_class_label"],
    values="land_cover_fraction",
    fill_value=0
)

wide_df.head()

buffer_radius_m                500                                            \
nlcd_class_label   cultivated_crops developed    forest grassland_herbaceous   
site_id trap_label                                                             
29221   Trap 101           0.461538  0.203215  0.171068             0.009185   
29222   Trap 102           0.613088  0.091848  0.071183             0.000000   
29224   Trap 104           0.442529  0.129885  0.186207             0.010345   
29225   Trap 105           0.780460  0.129885  0.003448             0.000000   
29226   Trap 106           0.589183  0.065593  0.128884             0.008055   

buffer_radius_m                                                  1000  \
nlcd_class_label   pasture_hay shrub_scrub     water  wetlands barren   
site_id trap_label                                                      
29221   Trap 101      0.000000    0.017222  0.011481  0.126292    0.0   
29222   Trap 102      0.001148    0.000000  0.029851  0.192882    0.0   
29224   Trap 104      0.001149    0.110345  0.018391  0.101149    0.0   
29225   Trap 105      0.000000    0.010345  0.000000  0.075862    0.0   
29226   Trap 106      0.005754    0.026467  0.032221  0.143843    0.0   

buffer_radius_m                                                               \
nlcd_class_label   cultivated_crops developed    forest grassland_herbaceous   
site_id trap_label                                                             
29221   Trap 101           0.499570  0.197878  0.128477             0.006596   
29222   Trap 102           0.625645  0.081182  0.077740             0.000574   
29224   Trap 104           0.317227  0.081680  0.254530             0.017544   
29225   Trap 105           0.711092  0.073947  0.034394             0.001146   
29226   Trap 106           0.582329  0.073723  0.172404             0.017785   

buffer_radius_m                                                 
nlcd_class_label   pasture_hay shrub_scrub     water  wetlands  
site_id trap_label                                              
29221   Trap 101      0.007169    0.014052  0.020075  0.126183  
29222   Trap 102      0.017499    0.002295  0.008893  0.186173  
29224   Trap 104      0.000575    0.134599  0.008053  0.185792  
29225   Trap 105      0.001146    0.005159  0.010032  0.163084  
29226   Trap 106      0.003155    0.020080  0.017212  0.113310

In [164]:
wide_df.columns

MultiIndex([( 500,     'cultivated_crops'),
            ( 500,            'developed'),
            ( 500,               'forest'),
            ( 500, 'grassland_herbaceous'),
            ( 500,          'pasture_hay'),
            ( 500,          'shrub_scrub'),
            ( 500,                'water'),
            ( 500,             'wetlands'),
            (1000,               'barren'),
            (1000,     'cultivated_crops'),
            (1000,            'developed'),
            (1000,               'forest'),
            (1000, 'grassland_herbaceous'),
            (1000,          'pasture_hay'),
            (1000,          'shrub_scrub'),
            (1000,                'water'),
            (1000,             'wetlands')],
           names=['buffer_radius_m', 'nlcd_class_label'])

In [165]:
# Flatten MultiIndex columns
wide_df.columns = [
    f"{group}_fraction_{radius}m"
    for radius, group in wide_df.columns
]
wide_df.columns

Index(['cultivated_crops_fraction_500m', 'developed_fraction_500m',
       'forest_fraction_500m', 'grassland_herbaceous_fraction_500m',
       'pasture_hay_fraction_500m', 'shrub_scrub_fraction_500m',
       'water_fraction_500m', 'wetlands_fraction_500m',
       'barren_fraction_1000m', 'cultivated_crops_fraction_1000m',
       'developed_fraction_1000m', 'forest_fraction_1000m',
       'grassland_herbaceous_fraction_1000m', 'pasture_hay_fraction_1000m',
       'shrub_scrub_fraction_1000m', 'water_fraction_1000m',
       'wetlands_fraction_1000m'],
      dtype='str')

In [166]:
# Bring site_id and trap_label back as columns
wide_df = wide_df.reset_index()

wide_df.head()

,site_id,trap_label,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,wetlands_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,0.461538,0.203215,0.171068,0.009185,0.000000,0.017222,0.011481,0.126292,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
1,29222,Trap 102,0.613088,0.091848,0.071183,0.000000,0.001148,0.000000,0.029851,0.192882,0.0,0.625645,0.081182,0.077740,0.000574,0.017499,0.002295,0.008893,0.186173
2,29224,Trap 104,0.442529,0.129885,0.186207,0.010345,0.001149,0.110345,0.018391,0.101149,0.0,0.317227,0.081680,0.254530,0.017544,0.000575,0.134599,0.008053,0.185792
3,29225,Trap 105,0.780460,0.129885,0.003448,0.000000,0.000000,0.010345,0.000000,0.075862,0.0,0.711092,0.073947,0.034394,0.001146,0.001146,0.005159,0.010032,0.163084
4,29226,Trap 106,0.589183,0.065593,0.128884,0.008055,0.005754,0.026467,0.032221,0.143843,0.0,0.582329,0.073723,0.172404,0.017785,0.003155,0.020080,0.017212,0.113310


In [167]:
wide_df["landcover_year_used"] = 2025
wide_df["landcover_source"] = "Annual_NLCD"

print(wide_df.shape)

print("Columns after flattening:")
print(wide_df.columns.to_list())

(31, 21)
Columns after flattening:
['site_id', 'trap_label', 'cultivated_crops_fraction_500m', 'developed_fraction_500m', 'forest_fraction_500m', 'grassland_herbaceous_fraction_500m', 'pasture_hay_fraction_500m', 'shrub_scrub_fraction_500m', 'water_fraction_500m', 'wetlands_fraction_500m', 'barren_fraction_1000m', 'cultivated_crops_fraction_1000m', 'developed_fraction_1000m', 'forest_fraction_1000m', 'grassland_herbaceous_fraction_1000m', 'pasture_hay_fraction_1000m', 'shrub_scrub_fraction_1000m', 'water_fraction_1000m', 'wetlands_fraction_1000m', 'landcover_year_used', 'landcover_source']


In [169]:
wide_df.columns.to_list()

['site_id',
 'trap_label',
 'cultivated_crops_fraction_500m',
 'developed_fraction_500m',
 'forest_fraction_500m',
 'grassland_herbaceous_fraction_500m',
 'pasture_hay_fraction_500m',
 'shrub_scrub_fraction_500m',
 'water_fraction_500m',
 'wetlands_fraction_500m',
 'barren_fraction_1000m',
 'cultivated_crops_fraction_1000m',
 'developed_fraction_1000m',
 'forest_fraction_1000m',
 'grassland_herbaceous_fraction_1000m',
 'pasture_hay_fraction_1000m',
 'shrub_scrub_fraction_1000m',
 'water_fraction_1000m',
 'wetlands_fraction_1000m',
 'landcover_year_used',
 'landcover_source']

In [170]:
fraction_cols = [
    col for col in wide_df.columns
    if col.endswith("500m") or col.endswith("1000m")
]

print("\nFraction column ranges:")
display(
    wide_df[fraction_cols]
    .agg(["min", "max"])  # compute both min and max for each column
    .T                    # transpose the result
    .sort_index()
)



Fraction column ranges:


,min,max
barren_fraction_1000m,0.000000,0.000575
cultivated_crops_fraction_1000m,0.188809,0.711092
cultivated_crops_fraction_500m,0.187067,0.784156
developed_fraction_1000m,0.022114,0.264901
developed_fraction_500m,0.029919,0.221330
forest_fraction_1000m,0.034394,0.306169
forest_fraction_500m,0.003448,0.412170
grassland_herbaceous_fraction_1000m,0.000287,0.028129
grassland_herbaceous_fraction_500m,0.000000,0.110599
pasture_hay_fraction_1000m,0.000000,0.143759


In [171]:
display(
    wide_df[
        [
            "site_id",
            "trap_label",
            "cultivated_crops_fraction_500m",
            "cultivated_crops_fraction_1000m"
        ]
    ]
    .sort_values("cultivated_crops_fraction_500m", ascending=False)
    .head()
)

,site_id,trap_label,cultivated_crops_fraction_500m,cultivated_crops_fraction_1000m
23,31144,Mike Horne Rd,0.784156,0.562931
25,31565,Shiver Rd.,0.783659,0.698736
3,29225,Trap 105,0.780460,0.711092
24,31147,Old Berlin,0.773196,0.651724
13,29235,Trap 115,0.761194,0.699971


In [172]:
expected_landcover_groups = [
    "water",
    "developed",
    "barren",
    "forest",
    "shrub_scrub",
    "grassland_herbaceous",
    "pasture_hay",
    "cultivated_crops",
    "wetlands"
]

expected_radii = [500, 1000]

for radius in expected_radii:
    for group in expected_landcover_groups:
        column = f"{group}_fraction_{radius}m"
        if column not in wide_df.columns:
            wide_df[column] = 0.0

In [173]:
wide_df.columns

Index(['site_id', 'trap_label', 'cultivated_crops_fraction_500m',
       'developed_fraction_500m', 'forest_fraction_500m',
       'grassland_herbaceous_fraction_500m', 'pasture_hay_fraction_500m',
       'shrub_scrub_fraction_500m', 'water_fraction_500m',
       'wetlands_fraction_500m', 'barren_fraction_1000m',
       'cultivated_crops_fraction_1000m', 'developed_fraction_1000m',
       'forest_fraction_1000m', 'grassland_herbaceous_fraction_1000m',
       'pasture_hay_fraction_1000m', 'shrub_scrub_fraction_1000m',
       'water_fraction_1000m', 'wetlands_fraction_1000m',
       'landcover_year_used', 'landcover_source', 'barren_fraction_500m'],
      dtype='str')

In [174]:
wide_df.shape

(31, 22)

In [175]:
# Reorder columns
id_cols = [
    "site_id",
    "trap_label",
    "landcover_year_used",
    "landcover_source"
]

feature_cols = [
    f"{group}_fraction_{radius}m"
    for radius in expected_radii
    for group in expected_landcover_groups
]

reordered_cols = id_cols + feature_cols

wide_df = wide_df[reordered_cols]

print(wide_df.shape)
wide_df.columns

(31, 22)


Index(['site_id', 'trap_label', 'landcover_year_used', 'landcover_source',
       'water_fraction_500m', 'developed_fraction_500m',
       'barren_fraction_500m', 'forest_fraction_500m',
       'shrub_scrub_fraction_500m', 'grassland_herbaceous_fraction_500m',
       'pasture_hay_fraction_500m', 'cultivated_crops_fraction_500m',
       'wetlands_fraction_500m', 'water_fraction_1000m',
       'developed_fraction_1000m', 'barren_fraction_1000m',
       'forest_fraction_1000m', 'shrub_scrub_fraction_1000m',
       'grassland_herbaceous_fraction_1000m', 'pasture_hay_fraction_1000m',
       'cultivated_crops_fraction_1000m', 'wetlands_fraction_1000m'],
      dtype='str')

In [176]:
wide_df.head(10)

,site_id,trap_label,landcover_year_used,landcover_source,water_fraction_500m,developed_fraction_500m,barren_fraction_500m,forest_fraction_500m,shrub_scrub_fraction_500m,grassland_herbaceous_fraction_500m,...,wetlands_fraction_500m,water_fraction_1000m,developed_fraction_1000m,barren_fraction_1000m,forest_fraction_1000m,shrub_scrub_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,cultivated_crops_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2025,Annual_NLCD,0.011481,0.203215,0.0,0.171068,0.017222,0.009185,...,0.126292,0.020075,0.197878,0.0,0.128477,0.014052,0.006596,0.007169,0.499570,0.126183
1,29222,Trap 102,2025,Annual_NLCD,0.029851,0.091848,0.0,0.071183,0.000000,0.000000,...,0.192882,0.008893,0.081182,0.0,0.077740,0.002295,0.000574,0.017499,0.625645,0.186173
2,29224,Trap 104,2025,Annual_NLCD,0.018391,0.129885,0.0,0.186207,0.110345,0.010345,...,0.101149,0.008053,0.081680,0.0,0.254530,0.134599,0.017544,0.000575,0.317227,0.185792
3,29225,Trap 105,2025,Annual_NLCD,0.000000,0.129885,0.0,0.003448,0.010345,0.000000,...,0.075862,0.010032,0.073947,0.0,0.034394,0.005159,0.001146,0.001146,0.711092,0.163084
4,29226,Trap 106,2025,Annual_NLCD,0.032221,0.065593,0.0,0.128884,0.026467,0.008055,...,0.143843,0.017212,0.073723,0.0,0.172404,0.020080,0.017785,0.003155,0.582329,0.113310
5,29227,Trap 107,2025,Annual_NLCD,0.104238,0.159221,0.0,0.069874,0.009164,0.011455,...,0.043528,0.054996,0.162684,0.0,0.075151,0.004895,0.005183,0.000000,0.595163,0.101929
6,29228,Trap 108,2025,Annual_NLCD,0.021789,0.080275,0.0,0.302752,0.006881,0.000000,...,0.308486,0.029310,0.050575,0.0,0.279310,0.008046,0.002874,0.003448,0.296839,0.329598
7,29229,Trap 109,2025,Annual_NLCD,0.032110,0.080275,0.0,0.088303,0.000000,0.003440,...,0.072248,0.068293,0.096987,0.0,0.060545,0.002869,0.006600,0.143759,0.517360,0.103587
8,29230,Trap 110,2025,Annual_NLCD,0.013746,0.124857,0.0,0.176403,0.022910,0.000000,...,0.222222,0.014634,0.167001,0.0,0.255093,0.016930,0.020373,0.143185,0.188809,0.193974
9,29231,Trap 111,2025,Annual_NLCD,0.000000,0.118255,0.0,0.412170,0.024110,0.000000,...,0.090700,0.006887,0.167862,0.0,0.306169,0.018938,0.000861,0.018077,0.350359,0.130846


In [177]:
cols_500m = [
    col for col in wide_df.columns
    if col.endswith("500m")
]

cols_1000m = [
    col for col in wide_df.columns
    if col.endswith("1000m")
]

fraction_sum_500m = wide_df[cols_500m].sum(axis=1)
print(fraction_sum_500m.describe())
fraction_sum_1000m = wide_df[cols_1000m].sum(axis=1)
print(fraction_sum_1000m.describe())

count    3.100000e+01
mean     1.000000e+00
std      3.510833e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
count    3.100000e+01
mean     1.000000e+00
std      7.308383e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


### Expand NLCD land cover feature extraction to all the years

In [178]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_NLCD_DIR = DATA_DIR / "raw" / "nlcd"
INTERIM_GEOSPATIAL_DIR = DATA_DIR / "interim" / "geospatial"
PROCESSED_DIR = DATA_DIR / "processed"

BUFFER_PATH = INTERIM_GEOSPATIAL_DIR / "trap_buffers_500m_1km.gpkg"

YEARS_REQUIRED = list(range(2020, 2027))

nlcd_paths = {
    year: RAW_NLCD_DIR / str(year) / f"nlcd_landcover_{year}.tif"
    for year in range(2020, 2026)
}

nlcd_paths

{2020: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2020/nlcd_landcover_2020.tif'),
 2021: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2021/nlcd_landcover_2021.tif'),
 2022: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2022/nlcd_landcover_2022.tif'),
 2023: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2023/nlcd_landcover_2023.tif'),
 2024: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2024/nlcd_landcover_2024.tif'),
 2025: PosixPath('/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2025/nlcd_landcover_2025.tif')}

In [179]:
for year, path in nlcd_paths.items():
    print(year, path.exists(), path)

2020 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2020/nlcd_landcover_2020.tif
2021 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2021/nlcd_landcover_2021.tif
2022 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2022/nlcd_landcover_2022.tif
2023 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2023/nlcd_landcover_2023.tif
2024 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2024/nlcd_landcover_2024.tif
2025 True /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2025/nlcd_landcover_2025.tif


In [180]:
trap_week_df = pd.read_csv(TRAP_COUNT_PATH, parse_dates=["date_collected"])

buffers_gdf = gpd.read_file(BUFFER_PATH)

print("Weekly trap count df shape:", trap_week_df.shape)
print("Years in weekly trap count df:", trap_week_df["year"].unique())
print("Trap sites buffers geo df shape:", buffers_gdf.shape)
print("Buffers geo df CRS:", buffers_gdf.crs)
print("Buffer radius counts:")
print(buffers_gdf["buffer_radius_m"].value_counts().sort_index())

Weekly trap count df shape: (6121, 11)
Years in weekly trap count df: [2020 2021 2022 2023 2024 2025 2026]
Trap sites buffers geo df shape: (62, 9)
Buffers geo df CRS: EPSG:26917
Buffer radius counts:
buffer_radius_m
500     31
1000    31
Name: count, dtype: int64


#### Inspect each raster before extraction

In [182]:
raster_inventory = []

for year, path in nlcd_paths.items():
    if not path.exists():
        raster_inventory.append(
            {
                "landcover_year": str(year),
                "exists": False,
                "path": str(path),
                "crs": None,
                "resolution": None,
                "nodata": None,
                "dtype": None,
                "band_count": None,
            }
        )
        continue

    with rasterio.open(path) as src:
        raster_inventory.append(
            {
                "landcover_year": str(year),
                "exists": True,
                "path": str(path),
                "crs": str(src.crs),
                "resolution": src.res,
                "nodata": src.nodata,
                "dtype": src.dtypes[0],
                "band_count": src.count,
            }
        )

raster_inventory_df = pd.DataFrame(raster_inventory)
raster_inventory_df

,landcover_year,exists,path,crs,resolution,nodata,dtype,band_count
0,2020,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1
1,2021,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1
2,2022,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1
3,2023,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1
4,2024,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1
5,2025,True,/Users/gurjitsingh/Desktop/DS_and_AI_projects/...,"PROJCS[""AEA WGS84"",GEOGCS[""WGS 84"",DATU...","(30.0, 30.0)",250.0,uint8,1


#### Build the reusable class mapping

In [184]:
NLCD_GROUPS = {
    "water": [11],
    "developed": [21, 22, 23, 24],
    "barren": [31],
    "forest": [41, 42, 43],
    "shrub_scrub": [52],
    "grassland_herbaceous": [71],
    "pasture_hay": [81],
    "cultivated_crops": [82],
    "wetlands": [90, 95],
}

codes_to_groups = {}
for group, codes in NLCD_GROUPS.items():
    for code in codes:
        codes_to_groups[code] = group

codes_to_groups

{11: 'water',
 21: 'developed',
 22: 'developed',
 23: 'developed',
 24: 'developed',
 31: 'barren',
 41: 'forest',
 42: 'forest',
 43: 'forest',
 52: 'shrub_scrub',
 71: 'grassland_herbaceous',
 81: 'pasture_hay',
 82: 'cultivated_crops',
 90: 'wetlands',
 95: 'wetlands'}

In [185]:
EXPECTED_GROUPS = list(NLCD_GROUPS.keys())
EXPECTED_RADII = [500, 1000]

EXPECTED_FEATURE_COLUMNS = [
    f"{group}_fraction_{radius}m"
    for radius in EXPECTED_RADII
    for group in EXPECTED_GROUPS
]

EXPECTED_FEATURE_COLUMNS

['water_fraction_500m',
 'developed_fraction_500m',
 'barren_fraction_500m',
 'forest_fraction_500m',
 'shrub_scrub_fraction_500m',
 'grassland_herbaceous_fraction_500m',
 'pasture_hay_fraction_500m',
 'cultivated_crops_fraction_500m',
 'wetlands_fraction_500m',
 'water_fraction_1000m',
 'developed_fraction_1000m',
 'barren_fraction_1000m',
 'forest_fraction_1000m',
 'shrub_scrub_fraction_1000m',
 'grassland_herbaceous_fraction_1000m',
 'pasture_hay_fraction_1000m',
 'cultivated_crops_fraction_1000m',
 'wetlands_fraction_1000m']

#### Validate raster overlap for all years

In [193]:
validation_rows = []
for year, path in nlcd_paths.items():
    # Open the raster
    with rasterio.open(path) as src:
        # Get its CRS and bounds
        raster_crs = src.crs
        raster_bounds = src.bounds

        # Convert raster bounds to a polygon
        raster_bounds_gdf = gpd.GeoDataFrame(
            {"name": ["raster_bounds"]},
            geometry=[box(raster_bounds.left, raster_bounds.bottom, raster_bounds.right, raster_bounds.top)],
            crs=raster_crs
        )

        # Reproject the trap buffers to the raster's CRS
        buffers_raster_crs_gdf = buffers_gdf.to_crs(raster_crs)

        # build rows for a small validation table
        validation_rows.append(
            {
                "year": int(year),
                "total_buffers": len(buffers_raster_crs_gdf),
                "intersecting_buffers": buffers_raster_crs_gdf.intersects(raster_bounds_gdf.geometry.iloc[0]).sum(),
                "all_buffers_intersect": buffers_raster_crs_gdf.intersects(raster_bounds_gdf.geometry.iloc[0]).all()
            }
        )

overlap_validate_df = pd.DataFrame(validation_rows)
overlap_validate_df

,year,total_buffers,intersecting_buffers,all_buffers_intersect
0,2020,62,62,True
1,2021,62,62,True
2,2022,62,62,True
3,2023,62,62,True
4,2024,62,62,True
5,2025,62,62,True


#### Validate the actual NLCD class values in every raster year

In [224]:
summary_records = []
for year, path in nlcd_paths.items():
    with rasterio.open(path) as src:
        raster_crs = src.crs
        nodata = src.nodata

        buffers_for_raster = buffers_gdf.to_crs(raster_crs)

        combined_class_codes = []
        for _, row in buffers_for_raster.iterrows():
            geom = [row.geometry]

            out_image, _ = mask(
                src,
                geom,
                crop=True,
                filled=True,
                nodata=nodata
            )

            values = out_image[0].ravel()

            # Remove nodata values
            if nodata is not None:
                values = values[values != nodata]

            # Remove background values
            values = values[values != 0]

            # Compute the unique remaining NLCD class codes for the buffer
            unique_class_codes = np.unique(values).tolist()

            combined_class_codes = combined_class_codes + unique_class_codes

        # Calculate the year-wide unique class codes
        year_wise_unique_class_codes = np.unique(combined_class_codes)

        # Compare these codes against the class codes included in codes_to_groups dictionary
        mapped_class_codes = [
            int(code) for code in year_wise_unique_class_codes
            if code in codes_to_groups.keys()
            ]
        unmapped_class_codes = [
            int(code) for code in year_wise_unique_class_codes
            if code not in codes_to_groups.keys()
            ]

        # Build rows for summary table
        summary_records.append(
            {
                "landcover_year": int(year),
                "n_unique_classes": int(len(year_wise_unique_class_codes)),
                "mapped_class_codes": mapped_class_codes,
                "unmapped_class_codes": unmapped_class_codes
            }
        )

summary_records_df = pd.DataFrame(summary_records)
summary_records_df

,landcover_year,n_unique_classes,mapped_class_codes,unmapped_class_codes
0,2020,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]
1,2021,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]
2,2022,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]
3,2023,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]
4,2024,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]
5,2025,15,"[11, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 8...",[]


#### Feature extraction for each site and each year

In [229]:
extraction_rows = []
for year, path in nlcd_paths.items():
    # Open the raster one at a time
    with rasterio.open(path) as src:
        # Find its CRS and nodata value
        raster_crs = src.crs
        nodata = src.nodata

        # Reproject the buffer's CRS onto raster's CRS
        buffers_for_raster = buffers_gdf.to_crs(raster_crs)

        # Compute raster polygon corressponding to each site buffer
        for _, row in buffers_for_raster.iterrows():
            geom = [row.geometry]

            out_image = mask(
                src,
                geom,
                crop=True,
                filled=True,
                nodata=nodata
            )

            # Flatten the polygon's pixels
            code_values = out_image[0].ravel()

            # Remove the pixels with nodata values
            if nodata is not None:
                code_values = code_values[code_values != nodata]

            # Remove the pixels with background noise
            code_values = code_values[code_values != 0]

            # For each of the (site x buffer), compute unique landcover codes and their counts
            unique_codes, pixel_counts = np.unique(code_values, return_counts=True)

            # Total valid pixels for the (site x buffer)
            valid_pixels = code_values.shape[0]

            # Metadata regarding the site
            site_id = row["site_id"]
            trap_label = row["trap_label"]
            buffer_radius_m = row["buffer_radius_m"]

            # Build rows for the feature extraction dataframe
            for unique_code, pixel_count in zip(unique_codes, pixel_counts):
                extraction_rows.append(
                    {
                        "site_id": site_id,
                        "trap_label": trap_label,
                        "buffer_radius_m": buffer_radius_m,
                        "landcover_year": int(year),
                        "landcover_code": int(unique_code),
                        "pixel_count": int(pixel_count),
                        "total_valid_pixels": int(valid_pixels)
                    }
                )

extraction_df = pd.DataFrame(extraction_rows)
extraction_df["landcover_group"] = extraction_df["landcover_code"].map(codes_to_groups)
extraction_df.head(10)

,site_id,trap_label,buffer_radius_m,landcover_year,landcover_code,pixel_count,total_valid_pixels,landcover_group
0,29221,Trap 101,500,2020,11,7,871,water
1,29221,Trap 101,500,2020,21,104,871,developed
2,29221,Trap 101,500,2020,22,71,871,developed
3,29221,Trap 101,500,2020,42,150,871,forest
4,29221,Trap 101,500,2020,52,15,871,shrub_scrub
5,29221,Trap 101,500,2020,71,5,871,grassland_herbaceous
6,29221,Trap 101,500,2020,82,410,871,cultivated_crops
7,29221,Trap 101,500,2020,90,108,871,wetlands
8,29221,Trap 101,500,2020,95,1,871,wetlands
9,29222,Trap 102,500,2020,11,16,871,water


#### Aggregate the table from class codes to broad land cover groups

In [246]:
valid_pixel_counts_df = extraction_df[
    [
        "site_id", "trap_label", "buffer_radius_m", "landcover_year", 
        "total_valid_pixels"
    ]
]

valid_pixel_counts_df = valid_pixel_counts_df.drop_duplicates(
    subset=["site_id", "buffer_radius_m", "landcover_year"]
).reset_index().drop(columns="index")

print(valid_pixel_counts_df.shape)
display(valid_pixel_counts_df.head(10))

(372, 5)


,site_id,trap_label,buffer_radius_m,landcover_year,total_valid_pixels
0,29221,Trap 101,500,2020,871
1,29222,Trap 102,500,2020,871
2,29224,Trap 104,500,2020,870
3,29225,Trap 105,500,2020,870
4,29226,Trap 106,500,2020,869
5,29227,Trap 107,500,2020,873
6,29228,Trap 108,500,2020,872
7,29229,Trap 109,500,2020,872
8,29230,Trap 110,500,2020,873
9,29231,Trap 111,500,2020,871


In [245]:
grouped_pixel_count_df = extraction_df.groupby(
    ["site_id", "trap_label", "buffer_radius_m", "landcover_year", "landcover_group"],
    as_index=False
)["pixel_count"].sum()

print(grouped_pixel_count_df.shape)
display(grouped_pixel_count_df.head(10))

(2693, 6)


,site_id,trap_label,buffer_radius_m,landcover_year,landcover_group,pixel_count
0,29221,Trap 101,500,2020,cultivated_crops,410
1,29221,Trap 101,500,2020,developed,175
2,29221,Trap 101,500,2020,forest,150
3,29221,Trap 101,500,2020,grassland_herbaceous,5
4,29221,Trap 101,500,2020,shrub_scrub,15
5,29221,Trap 101,500,2020,water,7
6,29221,Trap 101,500,2020,wetlands,109
7,29221,Trap 101,500,2021,cultivated_crops,411
8,29221,Trap 101,500,2021,developed,174
9,29221,Trap 101,500,2021,forest,150


In [244]:
aggregated_df = grouped_pixel_count_df.merge(
    valid_pixel_counts_df,
    on=["site_id", "trap_label", "buffer_radius_m", "landcover_year"],
    how="left"
)

aggregated_df["fraction"] = aggregated_df["pixel_count"] / aggregated_df["total_valid_pixels"]

print(aggregated_df.shape)
display(aggregated_df.head(10))

(2693, 8)


,site_id,trap_label,buffer_radius_m,landcover_year,landcover_group,pixel_count,total_valid_pixels,fraction
0,29221,Trap 101,500,2020,cultivated_crops,410,871,0.470723
1,29221,Trap 101,500,2020,developed,175,871,0.200918
2,29221,Trap 101,500,2020,forest,150,871,0.172216
3,29221,Trap 101,500,2020,grassland_herbaceous,5,871,0.005741
4,29221,Trap 101,500,2020,shrub_scrub,15,871,0.017222
5,29221,Trap 101,500,2020,water,7,871,0.008037
6,29221,Trap 101,500,2020,wetlands,109,871,0.125144
7,29221,Trap 101,500,2021,cultivated_crops,411,871,0.471871
8,29221,Trap 101,500,2021,developed,174,871,0.199770
9,29221,Trap 101,500,2021,forest,150,871,0.172216


In [258]:
unique_combos= aggregated_df[["site_id", "landcover_year", "buffer_radius_m"]].drop_duplicates()
unique_combos.shape[0]

372

In [265]:
validation_df = aggregated_df.groupby(
    ["site_id", "trap_label", "buffer_radius_m", "landcover_year"],
    as_index=False
)["fraction"].sum()

validation_df["fraction"].describe()

count    3.720000e+02
mean     1.000000e+00
std      9.983525e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: fraction, dtype: float64

In [267]:
aggregated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2693 entries, 0 to 2692
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   site_id             2693 non-null   int64  
 1   trap_label          2693 non-null   str    
 2   buffer_radius_m     2693 non-null   int64  
 3   landcover_year      2693 non-null   int64  
 4   landcover_group     2693 non-null   str    
 5   pixel_count         2693 non-null   int64  
 6   total_valid_pixels  2693 non-null   int64  
 7   fraction            2693 non-null   float64
dtypes: float64(1), int64(5), str(2)
memory usage: 168.4 KB


In [268]:
aggregated_df[["landcover_group", "fraction", "total_valid_pixels"]].isna().sum()

landcover_group       0
fraction              0
total_valid_pixels    0
dtype: int64

#### Create a modeling ready wide table

In [309]:
aggregated_df_wide = aggregated_df.pivot_table(
    columns=["landcover_group", "buffer_radius_m"],
    values="fraction",
    index=["site_id", "trap_label", "landcover_year"],
    fill_value=0
)

aggregated_df_wide.head()

landcover_group                   barren cultivated_crops           developed  \
buffer_radius_m                     1000             500       1000      500    
site_id trap_label landcover_year                                               
29221   Trap 101   2020              0.0         0.470723  0.515056  0.200918   
                   2021              0.0         0.471871  0.509034  0.199770   
                   2022              0.0         0.471871  0.505592  0.200918   
                   2023              0.0         0.467279  0.504158  0.202067   
                   2024              0.0         0.466131  0.500717  0.202067   

landcover_group                                forest            \
buffer_radius_m                        1000      500       1000   
site_id trap_label landcover_year                                 
29221   Trap 101   2020            0.189561  0.172216  0.124749   
                   2021            0.189561  0.172216  0.126470   
                   2022            0.192429  0.172216  0.126757   
                   2023            0.193863  0.172216  0.127330   
                   2024            0.196157  0.171068  0.127330   

landcover_group                   grassland_herbaceous           pasture_hay  \
buffer_radius_m                                   500       1000        500    
site_id trap_label landcover_year                                              
29221   Trap 101   2020                       0.005741  0.004875         0.0   
                   2021                       0.006889  0.006883         0.0   
                   2022                       0.005741  0.006022         0.0   
                   2023                       0.009185  0.006883         0.0   
                   2024                       0.010333  0.006596         0.0   

landcover_group                             shrub_scrub               water  \
buffer_radius_m                        1000        500       1000      500    
site_id trap_label landcover_year                                             
29221   Trap 101   2020            0.001434    0.017222  0.015486  0.008037   
                   2021            0.005736    0.017222  0.015199  0.006889   
                   2022            0.006022    0.017222  0.014913  0.006889   
                   2023            0.006022    0.017222  0.014913  0.006889   
                   2024            0.007169    0.017222  0.014052  0.006889   

landcover_group                              wetlands            
buffer_radius_m                        1000      500       1000  
site_id trap_label landcover_year                                
29221   Trap 101   2020            0.020648  0.125144  0.128190  
                   2021            0.018067  0.125144  0.129051  
                   2022            0.020361  0.125144  0.127904  
                   2023            0.019501  0.125144  0.127330  
                   2024            0.020648  0.126292  0.127330

In [310]:
aggregated_df_wide.columns = [
    f"{group}_fraction_{radius}m"
    for group, radius in aggregated_df_wide.columns.tolist()
]

aggregated_df_wide = aggregated_df_wide.reset_index()

print(aggregated_df_wide.shape)
display(aggregated_df_wide.head(10))


(186, 20)


,site_id,trap_label,landcover_year,barren_fraction_1000m,cultivated_crops_fraction_500m,cultivated_crops_fraction_1000m,developed_fraction_500m,developed_fraction_1000m,forest_fraction_500m,forest_fraction_1000m,grassland_herbaceous_fraction_500m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_500m,pasture_hay_fraction_1000m,shrub_scrub_fraction_500m,shrub_scrub_fraction_1000m,water_fraction_500m,water_fraction_1000m,wetlands_fraction_500m,wetlands_fraction_1000m
0,29221,Trap 101,2020,0.0,0.470723,0.515056,0.200918,0.189561,0.172216,0.124749,0.005741,0.004875,0.0,0.001434,0.017222,0.015486,0.008037,0.020648,0.125144,0.128190
1,29221,Trap 101,2021,0.0,0.471871,0.509034,0.199770,0.189561,0.172216,0.126470,0.006889,0.006883,0.0,0.005736,0.017222,0.015199,0.006889,0.018067,0.125144,0.129051
2,29221,Trap 101,2022,0.0,0.471871,0.505592,0.200918,0.192429,0.172216,0.126757,0.005741,0.006022,0.0,0.006022,0.017222,0.014913,0.006889,0.020361,0.125144,0.127904
3,29221,Trap 101,2023,0.0,0.467279,0.504158,0.202067,0.193863,0.172216,0.127330,0.009185,0.006883,0.0,0.006022,0.017222,0.014913,0.006889,0.019501,0.125144,0.127330
4,29221,Trap 101,2024,0.0,0.466131,0.500717,0.202067,0.196157,0.171068,0.127330,0.010333,0.006596,0.0,0.007169,0.017222,0.014052,0.006889,0.020648,0.126292,0.127330
5,29221,Trap 101,2025,0.0,0.461538,0.499570,0.203215,0.197878,0.171068,0.128477,0.009185,0.006596,0.0,0.007169,0.017222,0.014052,0.011481,0.020075,0.126292,0.126183
6,29222,Trap 102,2020,0.0,0.618829,0.640275,0.089552,0.076879,0.066590,0.072863,0.001148,0.000574,0.0,0.006885,0.000000,0.002008,0.018370,0.006311,0.205511,0.194205
7,29222,Trap 102,2021,0.0,0.613088,0.632817,0.089552,0.076879,0.066590,0.073437,0.000000,0.000287,0.0,0.012048,0.000000,0.002582,0.029851,0.008893,0.200918,0.193058
8,29222,Trap 102,2022,0.0,0.611940,0.633391,0.089552,0.076879,0.068886,0.074010,0.000000,0.000287,0.0,0.011761,0.000000,0.002582,0.033295,0.010040,0.196326,0.191050
9,29222,Trap 102,2023,0.0,0.614237,0.633104,0.089552,0.077166,0.068886,0.074297,0.000000,0.000287,0.0,0.012048,0.000000,0.002582,0.032147,0.009753,0.195178,0.190763


In [312]:
# If any of the expected column is missing, then add that column and assign values to 0
for col in EXPECTED_FEATURE_COLUMNS:
    if col not in [
        column for column in aggregated_df_wide.columns
        if (column.endswith("500m") or column.endswith("1000m"))
        ]:
        aggregated_df_wide[col] = 0

aggregated_df_wide.shape

(186, 21)

In [313]:
aggregated_df_wide.head(10)

,site_id,trap_label,landcover_year,barren_fraction_1000m,cultivated_crops_fraction_500m,cultivated_crops_fraction_1000m,developed_fraction_500m,developed_fraction_1000m,forest_fraction_500m,forest_fraction_1000m,...,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_500m,pasture_hay_fraction_1000m,shrub_scrub_fraction_500m,shrub_scrub_fraction_1000m,water_fraction_500m,water_fraction_1000m,wetlands_fraction_500m,wetlands_fraction_1000m,barren_fraction_500m
0,29221,Trap 101,2020,0.0,0.470723,0.515056,0.200918,0.189561,0.172216,0.124749,...,0.004875,0.0,0.001434,0.017222,0.015486,0.008037,0.020648,0.125144,0.128190,0
1,29221,Trap 101,2021,0.0,0.471871,0.509034,0.199770,0.189561,0.172216,0.126470,...,0.006883,0.0,0.005736,0.017222,0.015199,0.006889,0.018067,0.125144,0.129051,0
2,29221,Trap 101,2022,0.0,0.471871,0.505592,0.200918,0.192429,0.172216,0.126757,...,0.006022,0.0,0.006022,0.017222,0.014913,0.006889,0.020361,0.125144,0.127904,0
3,29221,Trap 101,2023,0.0,0.467279,0.504158,0.202067,0.193863,0.172216,0.127330,...,0.006883,0.0,0.006022,0.017222,0.014913,0.006889,0.019501,0.125144,0.127330,0
4,29221,Trap 101,2024,0.0,0.466131,0.500717,0.202067,0.196157,0.171068,0.127330,...,0.006596,0.0,0.007169,0.017222,0.014052,0.006889,0.020648,0.126292,0.127330,0
5,29221,Trap 101,2025,0.0,0.461538,0.499570,0.203215,0.197878,0.171068,0.128477,...,0.006596,0.0,0.007169,0.017222,0.014052,0.011481,0.020075,0.126292,0.126183,0
6,29222,Trap 102,2020,0.0,0.618829,0.640275,0.089552,0.076879,0.066590,0.072863,...,0.000574,0.0,0.006885,0.000000,0.002008,0.018370,0.006311,0.205511,0.194205,0
7,29222,Trap 102,2021,0.0,0.613088,0.632817,0.089552,0.076879,0.066590,0.073437,...,0.000287,0.0,0.012048,0.000000,0.002582,0.029851,0.008893,0.200918,0.193058,0
8,29222,Trap 102,2022,0.0,0.611940,0.633391,0.089552,0.076879,0.068886,0.074010,...,0.000287,0.0,0.011761,0.000000,0.002582,0.033295,0.010040,0.196326,0.191050,0
9,29222,Trap 102,2023,0.0,0.614237,0.633104,0.089552,0.077166,0.068886,0.074297,...,0.000287,0.0,0.012048,0.000000,0.002582,0.032147,0.009753,0.195178,0.190763,0


In [314]:
print(aggregated_df_wide.shape)
print(aggregated_df_wide.columns.tolist())

(186, 21)
['site_id', 'trap_label', 'landcover_year', 'barren_fraction_1000m', 'cultivated_crops_fraction_500m', 'cultivated_crops_fraction_1000m', 'developed_fraction_500m', 'developed_fraction_1000m', 'forest_fraction_500m', 'forest_fraction_1000m', 'grassland_herbaceous_fraction_500m', 'grassland_herbaceous_fraction_1000m', 'pasture_hay_fraction_500m', 'pasture_hay_fraction_1000m', 'shrub_scrub_fraction_500m', 'shrub_scrub_fraction_1000m', 'water_fraction_500m', 'water_fraction_1000m', 'wetlands_fraction_500m', 'wetlands_fraction_1000m', 'barren_fraction_500m']


In [318]:
# Reorder columns
aggregated_df_wide = aggregated_df_wide[
    ["site_id", "trap_label", "landcover_year"] + 
    [col for col in aggregated_df_wide.columns if col.endswith("500m")] +
    [col for col in aggregated_df_wide.columns if col.endswith("1000m")]
    ]
aggregated_df_wide.head()

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2020,0.470723,0.200918,0.172216,0.005741,0.0,0.017222,0.008037,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.128190
1,29221,Trap 101,2021,0.471871,0.199770,0.172216,0.006889,0.0,0.017222,0.006889,...,0,0.0,0.509034,0.189561,0.126470,0.006883,0.005736,0.015199,0.018067,0.129051
2,29221,Trap 101,2022,0.471871,0.200918,0.172216,0.005741,0.0,0.017222,0.006889,...,0,0.0,0.505592,0.192429,0.126757,0.006022,0.006022,0.014913,0.020361,0.127904
3,29221,Trap 101,2023,0.467279,0.202067,0.172216,0.009185,0.0,0.017222,0.006889,...,0,0.0,0.504158,0.193863,0.127330,0.006883,0.006022,0.014913,0.019501,0.127330
4,29221,Trap 101,2024,0.466131,0.202067,0.171068,0.010333,0.0,0.017222,0.006889,...,0,0.0,0.500717,0.196157,0.127330,0.006596,0.007169,0.014052,0.020648,0.127330


In [319]:
aggregated_df_wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   site_id                              186 non-null    int64  
 1   trap_label                           186 non-null    str    
 2   landcover_year                       186 non-null    int64  
 3   cultivated_crops_fraction_500m       186 non-null    float64
 4   developed_fraction_500m              186 non-null    float64
 5   forest_fraction_500m                 186 non-null    float64
 6   grassland_herbaceous_fraction_500m   186 non-null    float64
 7   pasture_hay_fraction_500m            186 non-null    float64
 8   shrub_scrub_fraction_500m            186 non-null    float64
 9   water_fraction_500m                  186 non-null    float64
 10  wetlands_fraction_500m               186 non-null    float64
 11  barren_fraction_500m                 186 no

#### Include the year 2026 to the feature table 

In [330]:
# For 2026, we use the same NLCD features as for 2025
feature_values_for_2026 = aggregated_df_wide.loc[aggregated_df_wide["landcover_year"] == 2025, :]
feature_values_for_2026 = feature_values_for_2026.reset_index(drop=True)
feature_values_for_2026["landcover_year"] = 2026
feature_values_for_2026

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2026,0.461538,0.203215,0.171068,0.009185,0.000000,0.017222,0.011481,...,0,0.000000,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
1,29222,Trap 102,2026,0.613088,0.091848,0.071183,0.000000,0.001148,0.000000,0.029851,...,0,0.000000,0.625645,0.081182,0.077740,0.000574,0.017499,0.002295,0.008893,0.186173
2,29224,Trap 104,2026,0.442529,0.129885,0.186207,0.010345,0.001149,0.110345,0.018391,...,0,0.000000,0.317227,0.081680,0.254530,0.017544,0.000575,0.134599,0.008053,0.185792
3,29225,Trap 105,2026,0.780460,0.129885,0.003448,0.000000,0.000000,0.010345,0.000000,...,0,0.000000,0.711092,0.073947,0.034394,0.001146,0.001146,0.005159,0.010032,0.163084
4,29226,Trap 106,2026,0.589183,0.065593,0.128884,0.008055,0.005754,0.026467,0.032221,...,0,0.000000,0.582329,0.073723,0.172404,0.017785,0.003155,0.020080,0.017212,0.113310
5,29227,Trap 107,2026,0.602520,0.159221,0.069874,0.011455,0.000000,0.009164,0.104238,...,0,0.000000,0.595163,0.162684,0.075151,0.005183,0.000000,0.004895,0.054996,0.101929
6,29228,Trap 108,2026,0.279817,0.080275,0.302752,0.000000,0.000000,0.006881,0.021789,...,0,0.000000,0.296839,0.050575,0.279310,0.002874,0.003448,0.008046,0.029310,0.329598
7,29229,Trap 109,2026,0.634174,0.080275,0.088303,0.003440,0.089450,0.000000,0.032110,...,0,0.000000,0.517360,0.096987,0.060545,0.006600,0.143759,0.002869,0.068293,0.103587
8,29230,Trap 110,2026,0.398625,0.124857,0.176403,0.000000,0.041237,0.022910,0.013746,...,0,0.000000,0.188809,0.167001,0.255093,0.020373,0.143185,0.016930,0.014634,0.193974
9,29231,Trap 111,2026,0.354765,0.118255,0.412170,0.000000,0.000000,0.024110,0.000000,...,0,0.000000,0.350359,0.167862,0.306169,0.000861,0.018077,0.018938,0.006887,0.130846


In [352]:
# Concatenate 2026 features with the aggregated df
concatenated_df = pd.concat([aggregated_df_wide, feature_values_for_2026], ignore_index=True)
concatenated_df.shape

(217, 21)

In [353]:
concatenated_df.head(10)

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2020,0.470723,0.200918,0.172216,0.005741,0.0,0.017222,0.008037,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.128190
1,29221,Trap 101,2021,0.471871,0.199770,0.172216,0.006889,0.0,0.017222,0.006889,...,0,0.0,0.509034,0.189561,0.126470,0.006883,0.005736,0.015199,0.018067,0.129051
2,29221,Trap 101,2022,0.471871,0.200918,0.172216,0.005741,0.0,0.017222,0.006889,...,0,0.0,0.505592,0.192429,0.126757,0.006022,0.006022,0.014913,0.020361,0.127904
3,29221,Trap 101,2023,0.467279,0.202067,0.172216,0.009185,0.0,0.017222,0.006889,...,0,0.0,0.504158,0.193863,0.127330,0.006883,0.006022,0.014913,0.019501,0.127330
4,29221,Trap 101,2024,0.466131,0.202067,0.171068,0.010333,0.0,0.017222,0.006889,...,0,0.0,0.500717,0.196157,0.127330,0.006596,0.007169,0.014052,0.020648,0.127330
5,29221,Trap 101,2025,0.461538,0.203215,0.171068,0.009185,0.0,0.017222,0.011481,...,0,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
6,29222,Trap 102,2020,0.618829,0.089552,0.066590,0.001148,0.0,0.000000,0.018370,...,0,0.0,0.640275,0.076879,0.072863,0.000574,0.006885,0.002008,0.006311,0.194205
7,29222,Trap 102,2021,0.613088,0.089552,0.066590,0.000000,0.0,0.000000,0.029851,...,0,0.0,0.632817,0.076879,0.073437,0.000287,0.012048,0.002582,0.008893,0.193058
8,29222,Trap 102,2022,0.611940,0.089552,0.068886,0.000000,0.0,0.000000,0.033295,...,0,0.0,0.633391,0.076879,0.074010,0.000287,0.011761,0.002582,0.010040,0.191050
9,29222,Trap 102,2023,0.614237,0.089552,0.068886,0.000000,0.0,0.000000,0.032147,...,0,0.0,0.633104,0.077166,0.074297,0.000287,0.012048,0.002582,0.009753,0.190763


In [341]:
concatenated_df.tail(10)

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
207,31141,Hwy 319,2026,0.448394,0.129587,0.266055,0.000000,0.000000,0.000000,0.000000,...,0,0.000000,0.565604,0.070055,0.235142,0.000287,0.000000,0.013781,0.000000,0.115131
208,31142,Funston/Sale city,2026,0.577982,0.161697,0.011468,0.022936,0.138761,0.005734,0.014908,...,0,0.000287,0.569377,0.114048,0.047113,0.008331,0.121804,0.007756,0.015800,0.115484
209,31144,Mike Horne Rd,2026,0.784156,0.094145,0.006889,0.000000,0.002296,0.000000,0.003444,...,0,0.000000,0.562931,0.076437,0.139655,0.001724,0.001149,0.004023,0.028448,0.185632
210,31147,Old Berlin,2026,0.773196,0.115693,0.061856,0.000000,0.008018,0.005727,0.001145,...,0,0.000287,0.651724,0.082471,0.151149,0.000862,0.012069,0.004598,0.009483,0.087356
211,31565,Shiver Rd.,2026,0.783659,0.029919,0.095512,0.004603,0.000000,0.000000,0.040276,...,0,0.000000,0.698736,0.022114,0.098219,0.003446,0.022114,0.004595,0.027570,0.123205
212,31566,Melton Brinson Rd.,2026,0.554535,0.087256,0.164179,0.001148,0.072331,0.000000,0.000000,...,0,0.000000,0.489068,0.077963,0.271001,0.009494,0.069620,0.000863,0.000288,0.081703
213,31627,HWY 200,2026,0.187067,0.076212,0.050808,0.001155,0.122402,0.039261,0.001155,...,0,0.000000,0.291906,0.037600,0.111940,0.000287,0.048220,0.019518,0.000574,0.489954
214,32085,Chafin Rd,2026,0.721649,0.135166,0.035510,0.006873,0.000000,0.002291,0.009164,...,0,0.000000,0.696787,0.092369,0.092656,0.001721,0.000000,0.000861,0.004303,0.111302
215,32409,Morven Field,2026,0.627294,0.083716,0.008028,0.001147,0.061927,0.003440,0.000000,...,0,0.000000,0.683606,0.061441,0.038185,0.001723,0.084123,0.001436,0.003445,0.126041
216,32410,Dixie Field,2026,0.301843,0.103687,0.254608,0.110599,0.000000,0.008065,0.003456,...,0,0.000000,0.382032,0.103042,0.270953,0.028129,0.000000,0.010333,0.002583,0.202928


In [368]:
concatenated_df = concatenated_df.sort_values(["site_id", "landcover_year"])
concatenated_df = concatenated_df.reset_index(drop=True)
concatenated_df.head(20)

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2020,0.470723,0.200918,0.172216,0.005741,0.000000,0.017222,0.008037,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.128190
1,29221,Trap 101,2021,0.471871,0.199770,0.172216,0.006889,0.000000,0.017222,0.006889,...,0,0.0,0.509034,0.189561,0.126470,0.006883,0.005736,0.015199,0.018067,0.129051
2,29221,Trap 101,2022,0.471871,0.200918,0.172216,0.005741,0.000000,0.017222,0.006889,...,0,0.0,0.505592,0.192429,0.126757,0.006022,0.006022,0.014913,0.020361,0.127904
3,29221,Trap 101,2023,0.467279,0.202067,0.172216,0.009185,0.000000,0.017222,0.006889,...,0,0.0,0.504158,0.193863,0.127330,0.006883,0.006022,0.014913,0.019501,0.127330
4,29221,Trap 101,2024,0.466131,0.202067,0.171068,0.010333,0.000000,0.017222,0.006889,...,0,0.0,0.500717,0.196157,0.127330,0.006596,0.007169,0.014052,0.020648,0.127330
5,29221,Trap 101,2025,0.461538,0.203215,0.171068,0.009185,0.000000,0.017222,0.011481,...,0,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
6,29221,Trap 101,2026,0.461538,0.203215,0.171068,0.009185,0.000000,0.017222,0.011481,...,0,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
7,29222,Trap 102,2020,0.618829,0.089552,0.066590,0.001148,0.000000,0.000000,0.018370,...,0,0.0,0.640275,0.076879,0.072863,0.000574,0.006885,0.002008,0.006311,0.194205
8,29222,Trap 102,2021,0.613088,0.089552,0.066590,0.000000,0.000000,0.000000,0.029851,...,0,0.0,0.632817,0.076879,0.073437,0.000287,0.012048,0.002582,0.008893,0.193058
9,29222,Trap 102,2022,0.611940,0.089552,0.068886,0.000000,0.000000,0.000000,0.033295,...,0,0.0,0.633391,0.076879,0.074010,0.000287,0.011761,0.002582,0.010040,0.191050


In [369]:
new_rows = []
for _, row in concatenated_df.iterrows():
    if row["landcover_year"] != 2026:
        row["landcover_year_used"] = row["landcover_year"]
        new_rows.append(row)
    else:
        row["landcover_year_used"] = 2025
        new_rows.append(row)

feature_table_df = pd.DataFrame(new_rows)
feature_table_df.tail(10)

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m,landcover_year_used
207,32409,Morven Field,2024,0.627294,0.083716,0.008028,0.001147,0.060780,0.003440,0.001147,...,0.0,0.683893,0.061441,0.038185,0.001436,0.083549,0.001436,0.003732,0.126328,2024
208,32409,Morven Field,2025,0.627294,0.083716,0.008028,0.001147,0.061927,0.003440,0.000000,...,0.0,0.683606,0.061441,0.038185,0.001723,0.084123,0.001436,0.003445,0.126041,2025
209,32409,Morven Field,2026,0.627294,0.083716,0.008028,0.001147,0.061927,0.003440,0.000000,...,0.0,0.683606,0.061441,0.038185,0.001723,0.084123,0.001436,0.003445,0.126041,2025
210,32410,Dixie Field,2020,0.276498,0.102535,0.345622,0.054147,0.000000,0.000000,0.000000,...,0.0,0.377153,0.102468,0.295637,0.013777,0.000000,0.007176,0.000287,0.203502,2020
211,32410,Dixie Field,2021,0.308756,0.103687,0.308756,0.055300,0.000000,0.002304,0.000000,...,0.0,0.384041,0.102755,0.287313,0.014064,0.000000,0.007463,0.000574,0.203789,2021
212,32410,Dixie Field,2022,0.308756,0.103687,0.296083,0.066820,0.000000,0.003456,0.003456,...,0.0,0.383467,0.103042,0.282147,0.017222,0.000000,0.008898,0.002296,0.202928,2022
213,32410,Dixie Field,2023,0.317972,0.103687,0.258065,0.095622,0.000000,0.003456,0.003456,...,0.0,0.385476,0.103042,0.273249,0.024110,0.000000,0.008611,0.002583,0.202928,2023
214,32410,Dixie Field,2024,0.305300,0.103687,0.253456,0.109447,0.000000,0.006912,0.004608,...,0.0,0.382319,0.103042,0.271814,0.027555,0.000000,0.009759,0.002870,0.202641,2024
215,32410,Dixie Field,2025,0.301843,0.103687,0.254608,0.110599,0.000000,0.008065,0.003456,...,0.0,0.382032,0.103042,0.270953,0.028129,0.000000,0.010333,0.002583,0.202928,2025
216,32410,Dixie Field,2026,0.301843,0.103687,0.254608,0.110599,0.000000,0.008065,0.003456,...,0.0,0.382032,0.103042,0.270953,0.028129,0.000000,0.010333,0.002583,0.202928,2025


In [370]:
# Add the year_matched boolean feature
feature_table_df["year_matched"] = [
    year == year_used
    for year, year_used in zip(
        feature_table_df["landcover_year"], 
        feature_table_df["landcover_year_used"]
        )
]
feature_table_df.head(10)

,site_id,trap_label,landcover_year,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,pasture_hay_fraction_500m,shrub_scrub_fraction_500m,water_fraction_500m,...,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m,landcover_year_used,year_matched
0,29221,Trap 101,2020,0.470723,0.200918,0.172216,0.005741,0.0,0.017222,0.008037,...,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.128190,2020,True
1,29221,Trap 101,2021,0.471871,0.199770,0.172216,0.006889,0.0,0.017222,0.006889,...,0.509034,0.189561,0.126470,0.006883,0.005736,0.015199,0.018067,0.129051,2021,True
2,29221,Trap 101,2022,0.471871,0.200918,0.172216,0.005741,0.0,0.017222,0.006889,...,0.505592,0.192429,0.126757,0.006022,0.006022,0.014913,0.020361,0.127904,2022,True
3,29221,Trap 101,2023,0.467279,0.202067,0.172216,0.009185,0.0,0.017222,0.006889,...,0.504158,0.193863,0.127330,0.006883,0.006022,0.014913,0.019501,0.127330,2023,True
4,29221,Trap 101,2024,0.466131,0.202067,0.171068,0.010333,0.0,0.017222,0.006889,...,0.500717,0.196157,0.127330,0.006596,0.007169,0.014052,0.020648,0.127330,2024,True
5,29221,Trap 101,2025,0.461538,0.203215,0.171068,0.009185,0.0,0.017222,0.011481,...,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183,2025,True
6,29221,Trap 101,2026,0.461538,0.203215,0.171068,0.009185,0.0,0.017222,0.011481,...,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183,2025,False
7,29222,Trap 102,2020,0.618829,0.089552,0.066590,0.001148,0.0,0.000000,0.018370,...,0.640275,0.076879,0.072863,0.000574,0.006885,0.002008,0.006311,0.194205,2020,True
8,29222,Trap 102,2021,0.613088,0.089552,0.066590,0.000000,0.0,0.000000,0.029851,...,0.632817,0.076879,0.073437,0.000287,0.012048,0.002582,0.008893,0.193058,2021,True
9,29222,Trap 102,2022,0.611940,0.089552,0.068886,0.000000,0.0,0.000000,0.033295,...,0.633391,0.076879,0.074010,0.000287,0.011761,0.002582,0.010040,0.191050,2022,True


In [371]:
# Add a column representing source of landcover
feature_table_df["landcover_source"] = "Annual_NLCD"

In [ ]:
# Rearrange the columns
id_cols = [
    "site_id", "trap_label", "landcover_year", "landcover_year_used", 
    "year_matched", "landcover_source"
    ]

feature_table_df = feature_table_df[
    id_cols +
    [col for col in feature_table_df.columns.to_list() if col not in id_cols]
]

# Rename the landcover_year column
feature_table_df = feature_table_df.rename(columns={"landcover_year": "trap_year"})

feature_table_df.head(10)

,site_id,trap_label,trap_year,landcover_year_used,year_matched,landcover_source,cultivated_crops_fraction_500m,developed_fraction_500m,forest_fraction_500m,grassland_herbaceous_fraction_500m,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,Trap 101,2020,2020,True,Annual_NLCD,0.470723,0.200918,0.172216,0.005741,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.128190
1,29221,Trap 101,2021,2021,True,Annual_NLCD,0.471871,0.199770,0.172216,0.006889,...,0,0.0,0.509034,0.189561,0.126470,0.006883,0.005736,0.015199,0.018067,0.129051
2,29221,Trap 101,2022,2022,True,Annual_NLCD,0.471871,0.200918,0.172216,0.005741,...,0,0.0,0.505592,0.192429,0.126757,0.006022,0.006022,0.014913,0.020361,0.127904
3,29221,Trap 101,2023,2023,True,Annual_NLCD,0.467279,0.202067,0.172216,0.009185,...,0,0.0,0.504158,0.193863,0.127330,0.006883,0.006022,0.014913,0.019501,0.127330
4,29221,Trap 101,2024,2024,True,Annual_NLCD,0.466131,0.202067,0.171068,0.010333,...,0,0.0,0.500717,0.196157,0.127330,0.006596,0.007169,0.014052,0.020648,0.127330
5,29221,Trap 101,2025,2025,True,Annual_NLCD,0.461538,0.203215,0.171068,0.009185,...,0,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
6,29221,Trap 101,2026,2025,False,Annual_NLCD,0.461538,0.203215,0.171068,0.009185,...,0,0.0,0.499570,0.197878,0.128477,0.006596,0.007169,0.014052,0.020075,0.126183
7,29222,Trap 102,2020,2020,True,Annual_NLCD,0.618829,0.089552,0.066590,0.001148,...,0,0.0,0.640275,0.076879,0.072863,0.000574,0.006885,0.002008,0.006311,0.194205
8,29222,Trap 102,2021,2021,True,Annual_NLCD,0.613088,0.089552,0.066590,0.000000,...,0,0.0,0.632817,0.076879,0.073437,0.000287,0.012048,0.002582,0.008893,0.193058
9,29222,Trap 102,2022,2022,True,Annual_NLCD,0.611940,0.089552,0.068886,0.000000,...,0,0.0,0.633391,0.076879,0.074010,0.000287,0.011761,0.002582,0.010040,0.191050


In [373]:
feature_table_df.shape

(217, 24)

#### Merge the landcover feature table with weekly whitefly counts data table

In [ ]:
# Weekly whitefly count dataset
trap_count_df.head()

,site_id,year,plotted_date,date_collected,whitefly_count,trap_label,latitude,longitude,status,first_report,last_report
0,29221,2020,2024-04-22,2020-04-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22
1,29221,2020,2024-04-28,2020-04-28,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22
2,29221,2020,2024-05-06,2020-05-06,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22
3,29221,2020,2024-05-15,2020-05-15,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22
4,29221,2020,2024-05-22,2020-05-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22


In [376]:
merged_dataset = pd.merge(
    trap_count_df,
    feature_table_df,
    how="left",
    left_on=["site_id", "year"],
    right_on=["site_id", "trap_year"]
)

print("Total row count:", merged_dataset.shape[0])
print("\nTotal duplicate rows:", merged_dataset.duplicated().sum())
print("\nMissing values:", merged_dataset.isna().sum())
display(merged_dataset.head(10))

Total row count: 6121

Total duplicate rows: 0

Missing values: site_id                                 0
year                                    0
plotted_date                            0
date_collected                          0
whitefly_count                          0
trap_label_x                            0
latitude                                0
longitude                               0
status                                  0
first_report                            0
last_report                            60
trap_label_y                            0
trap_year                               0
landcover_year_used                     0
year_matched                            0
landcover_source                        0
cultivated_crops_fraction_500m          0
developed_fraction_500m                 0
forest_fraction_500m                    0
grassland_herbaceous_fraction_500m      0
pasture_hay_fraction_500m               0
shrub_scrub_fraction_500m               0
water_fracti

,site_id,year,plotted_date,date_collected,whitefly_count,trap_label_x,latitude,longitude,status,first_report,...,barren_fraction_500m,barren_fraction_1000m,cultivated_crops_fraction_1000m,developed_fraction_1000m,forest_fraction_1000m,grassland_herbaceous_fraction_1000m,pasture_hay_fraction_1000m,shrub_scrub_fraction_1000m,water_fraction_1000m,wetlands_fraction_1000m
0,29221,2020,2024-04-22,2020-04-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
1,29221,2020,2024-04-28,2020-04-28,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
2,29221,2020,2024-05-06,2020-05-06,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
3,29221,2020,2024-05-15,2020-05-15,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
4,29221,2020,2024-05-22,2020-05-22,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
5,29221,2020,2024-05-28,2020-05-28,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
6,29221,2020,2024-06-12,2020-06-12,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
7,29221,2020,2024-06-19,2020-06-19,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
8,29221,2020,2024-06-26,2020-06-26,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819
9,29221,2020,2024-07-03,2020-07-03,0,Trap 101,31.46603,-83.465008,Active,2020-04-22,...,0,0.0,0.515056,0.189561,0.124749,0.004875,0.001434,0.015486,0.020648,0.12819


In [377]:
(merged_dataset["trap_label_x"] == merged_dataset["trap_label_y"]).all()

np.True_